# 🎭 화상 마피아 게임 - AI 기능 실현성 프로토타입

이 노트북은 **팀 개발 시작 전 "이 AI 기능들을 실제로 구현할 수 있는가?"** 를 검증하기 위한 것입니다.
각 섹션은 **독립적으로** 실행할 수 있으니, 필요한 부분만 골라 돌려보세요.

| 섹션 | 검증 대상 | 핵심 라이브러리 | Colab 실행 |
|------|-----------|-----------------|-----------|
| 1 | **STT** (음성→텍스트, 화자별) | `faster-whisper` | ✅ (GPU 권장) |
| 2 | **로그 분석 LLM** (조간신문/심판/진술) | OpenAI or Anthropic API | ✅ (API 키 필요) |
| 3 | **얼굴/표정 분석** (blendshape → 긴장도) | `mediapipe`, `deepface` | ✅ (이미지/영상 업로드) |
| 4 | **rPPG 심박 추정** (연출용) | OpenCV heuristic | ✅ (영상 업로드) |

> ⚙️ **Colab 팁**: `런타임 → 런타임 유형 변경 → GPU(T4)` 로 설정하면 STT가 훨씬 빠릅니다.

> ⚠️ **실시간(웹캠 스트리밍)** 은 Colab에서 재현이 까다롭습니다. 실서비스에서는
> MediaPipe는 **브라우저 JS**로, 심화 분석은 **서버(FastAPI)** 에서 처리하는 구조가 맞습니다.
> 이 노트북은 "알고리즘이 원하는 출력을 내는가"를 **오프라인(업로드 파일)** 으로 검증합니다.


---
## 1️⃣ STT — 음성을 텍스트로 (화자 로그의 원천)

마피아 게임 로그 분석의 전제 조건: **"누가 무엇을 말했는가"**.
여기서는 `faster-whisper`로 한국어 전사 + 타임스탬프를 확인합니다.

> 💡 **화자 분리(diarization)** 는 Whisper 단독으로 안 됩니다. 실서비스에서는
> **WebRTC에서 참가자별 오디오 트랙을 따로 받아** 각 트랙을 개별 전사하는 것이 가장 정확합니다.
> (아래는 단일 오디오 파일 전사 데모)

In [ ]:
# STT 의존성 설치
!pip -q install faster-whisper

In [ ]:
# 오디오 업로드 (mp3/wav/m4a). 짧은 한국어 발화 샘플 하나면 충분합니다.
from google.colab import files
print("오디오 파일을 업로드하세요 (예: 30초 한국어 발화)")
uploaded = files.upload()
AUDIO_PATH = list(uploaded.keys())[0]
print("업로드됨:", AUDIO_PATH)

In [ ]:
from faster_whisper import WhisperModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"device={device}, compute_type={compute}")

# large-v3 가 정확하지만 무겁습니다. 프로토타입은 medium 도 충분.
model = WhisperModel("large-v3", device=device, compute_type=compute)

segments, info = model.transcribe(AUDIO_PATH, language="ko", vad_filter=True)
print(f"감지 언어: {info.language} (p={info.language_probability:.2f})\n")

# 게임 로그 형태로 출력해보기
transcript = []
for seg in segments:
    line = {"start": round(seg.start, 2), "end": round(seg.end, 2), "text": seg.text.strip()}
    transcript.append(line)
    print(f"[{line['start']:6.2f}s - {line['end']:6.2f}s] {line['text']}")

print("\n✅ 판단: 위 전사가 실제 발화와 얼마나 일치하는지 확인하세요.")

**✔ 실현성 체크포인트**
- 한국어 정확도가 게임 로그로 쓸 만한가? (부정확하면 Naver CLOVA Speech 등 유료 STT 검토)
- 실서비스는 **참가자별 트랙 전사**로 `userId`를 확실히 붙이는 것을 전제로 설계하세요.

---
## 2️⃣ 로그 분석 LLM — 조간신문 / AI 심판 / 진술 분석

세 기능 모두 **같은 발화 로그**를 입력으로 씁니다. 그래서 **로그 스키마 설계가 가장 중요**합니다.
아래는 스키마 정의 → 샘플 로그 → 3가지 프롬프트를 실제 LLM으로 호출하는 데모입니다.

In [ ]:
# 발화 로그 스키마 (실서비스 DB/JSON 설계의 기준안)
from dataclasses import dataclass, asdict
from typing import Literal
import json

@dataclass
class Utterance:
    turn: int
    day: int
    phase: Literal["day_discussion", "last_defense", "night"]
    user_id: str
    nickname: str
    text: str
    timestamp: float  # 게임 시작 후 경과 초

@dataclass
class GameState:
    day: int
    alive: list          # 생존자 nickname
    dead: list           # [{"nickname":..., "role":..., "day":...}]
    votes: dict          # {"투표대상": 득표수}

# ---- 샘플 게임 로그 (3일차 낮 토론) ----
logs = [
    Utterance(1, 3, "day_discussion", "u1", "지훈", "어제 밤에 민수가 죽었는데, 나는 영희가 계속 말을 돌리는 게 수상해.", 610.0),
    Utterance(2, 3, "day_discussion", "u2", "영희", "내가 왜? 나는 어제 지훈이가 투표를 갑자기 바꾼 게 더 이상했어.", 625.0),
    Utterance(3, 3, "day_discussion", "u3", "철수", "둘 다 진정해. 근데 영희 말도 일리는 있어. 지훈이 투표 왜 바꿨어?", 640.0),
    Utterance(4, 3, "day_discussion", "u1", "지훈", "아 그건... 그냥 분위기 보고 바꾼 거야. 별 의미 없어.", 652.0),
    Utterance(5, 3, "day_discussion", "u4", "영수", "지훈이 지금 되게 방어적이네. 나는 지훈이 의심스러움.", 668.0),
]

state = GameState(
    day=3,
    alive=["지훈", "영희", "철수", "영수"],
    dead=[{"nickname": "민수", "role": "시민", "day": 2}],
    votes={"지훈": 2, "영희": 1},
)

log_text = "\n".join(f"[{u.nickname}] {u.text}" for u in logs)
print(log_text)

In [ ]:
# LLM 클라이언트 설정 — OpenAI 또는 Anthropic 중 택1
# (둘 중 가지고 있는 API 키를 입력하세요)
import getpass, os

PROVIDER = "openai"   # "openai" 또는 "anthropic"

if PROVIDER == "openai":
    !pip -q install openai
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
    from openai import OpenAI
    _client = OpenAI()
    def ask_llm(system, user):
        r = _client.chat.completions.create(
            model="gpt-4o-mini",  # 프로토타입용. 실서비스는 품질 보고 상향
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.7,
        )
        return r.choices[0].message.content
else:
    !pip -q install anthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key: ")
    import anthropic
    _client = anthropic.Anthropic()
    def ask_llm(system, user):
        r = _client.messages.create(
            model="claude-sonnet-4-5",  # 최신 모델 id는 공식 문서에서 확인 후 교체
            max_tokens=1024, system=system,
            messages=[{"role":"user","content":user}],
        )
        return r.content[0].text

print("LLM 클라이언트 준비 완료:", PROVIDER)

In [ ]:
# (A) 조간신문 발행
sys_news = "너는 마피아 게임의 '조간신문' 기자다. 밤사이 사건과 낮 토론을 극적이고 몰입감 있게, 단 중립적으로 요약한다. 6문장 이내."
user_news = f"""[게임 상태]
- {state.day}일차 아침
- 사망: {state.dead}
- 생존: {state.alive}

[전날 낮 토론 로그]
{log_text}

위를 바탕으로 오늘자 마피아 조간신문 기사를 작성하라."""

print("📰 === 조간신문 ===\n")
print(ask_llm(sys_news, user_news))

In [ ]:
# (B) AI 심판 — 마피아 의심도 추정
sys_judge = ("너는 마피아 게임의 AI 심판이다. 발언의 논리적 모순, 회피, 방어성, "
             "투표 번복 등을 근거로 각 생존자의 '마피아 의심도'를 0~100으로 추정한다. "
             "반드시 근거를 함께 제시하고, 확정이 아닌 '추정'임을 명시한다. JSON으로 출력.")
user_judge = f"""[생존자] {state.alive}
[투표 현황] {state.votes}
[낮 토론 로그]
{log_text}

각 생존자별 {{"nickname":..., "suspicion":0-100, "reason":...}} 배열로 출력하라."""

print("⚖️ === AI 심판 의심도 ===\n")
print(ask_llm(sys_judge, user_judge))

In [ ]:
# (C) 아이템: 특정 인물 진술 집중 분석
TARGET = "지훈"
sys_stmt = ("너는 마피아 게임의 진술 분석 도구다. 지정된 인물의 발언만 모아 "
            "일관성, 모순, 감정 변화, 방어 패턴을 분석한다. 게임적 재미를 위한 추정임을 명시.")
target_logs = "\n".join(f"[{u.nickname}] {u.text}" for u in logs if u.nickname == TARGET)
user_stmt = f"""분석 대상: {TARGET}
[{TARGET}의 발언들]
{target_logs}

[전체 맥락]
{log_text}

{TARGET}의 진술을 분석하라."""

print(f"🔍 === {TARGET} 진술 분석 ===\n")
print(ask_llm(sys_stmt, user_stmt))

**✔ 실현성 체크포인트**
- 세 기능 모두 **동일한 로그 스키마**에서 나온다는 점 확인 → DB 설계 시 `Utterance` 구조가 핵심.
- 프롬프트만 바꾸면 기능이 늘어난다 → 확장성 좋음.
- 비용: 실시간이 아니라 **국면 전환 시점(아침/투표 후)** 에만 호출하므로 API 비용 관리 가능.

---
## 3️⃣ 얼굴/표정 분석 — 최후 변론 확대 화면

MediaPipe **Face Landmarker**의 52개 blendshape로 표정을 수치화하고,
DeepFace로 감정을 분류합니다. blendshape 조합으로 **"긴장/동요 지수"** 를 만들어 봅니다.

> ⚠️ **중요**: 표정 기반 '거짓말 탐지'는 과학적으로 검증되지 않았습니다.
> 아래 지표는 **오락용 연출("AI 동요 지수")** 로만 사용하고, 실제 진실 판정처럼 포장하지 마세요.

In [ ]:
!pip -q install mediapipe deepface
# MediaPipe blendshape 모델 다운로드
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print("설치 완료")

In [ ]:
# 얼굴 정면 이미지 업로드 (최후 변론 프레임 캡처 가정)
from google.colab import files
import cv2, numpy as np
print("얼굴 정면 이미지를 업로드하세요")
up = files.upload()
IMG_PATH = list(up.keys())[0]
img = cv2.imread(IMG_PATH)
print("이미지 크기:", img.shape)

In [ ]:
# MediaPipe Face Landmarker: blendshape 추출
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base = python.BaseOptions(model_asset_path="face_landmarker.task")
opts = vision.FaceLandmarkerOptions(base_options=base,
                                    output_face_blendshapes=True,
                                    num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(opts)

mp_img = mp.Image.create_from_file(IMG_PATH)
res = landmarker.detect(mp_img)

if not res.face_blendshapes:
    print("⚠️ 얼굴을 찾지 못했습니다. 정면 얼굴 이미지로 다시 시도하세요.")
else:
    bs = {c.category_name: c.score for c in res.face_blendshapes[0]}
    # 관심 지표 몇 개
    watch = ["eyeBlinkLeft","eyeBlinkRight","browDownLeft","browDownRight",
             "mouthPressLeft","mouthPressRight","jawOpen","eyeLookOutLeft","eyeLookOutRight"]
    print("주요 blendshape:")
    for k in watch:
        print(f"  {k:18s}: {bs.get(k,0):.3f}")

In [ ]:
# "AI 동요 지수" 휴리스틱 (연출용) — blendshape 조합으로 스코어링
def agitation_index(bs):
    blink   = (bs.get("eyeBlinkLeft",0)+bs.get("eyeBlinkRight",0))/2
    brow    = (bs.get("browDownLeft",0)+bs.get("browDownRight",0))/2   # 찡그림
    press   = (bs.get("mouthPressLeft",0)+bs.get("mouthPressRight",0))/2 # 입 앙다뭄
    gaze    = (bs.get("eyeLookOutLeft",0)+bs.get("eyeLookOutRight",0))/2 # 시선 회피
    score = 100*(0.30*blink + 0.25*brow + 0.25*press + 0.20*gaze)
    return round(min(score,100), 1)

if res.face_blendshapes:
    idx = agitation_index(bs)
    print(f"🎭 AI 동요 지수(연출용): {idx} / 100")
    print("   (깜빡임·찡그림·입 앙다뭄·시선회피 가중합. 과학적 진실판정 아님)")

In [ ]:
# DeepFace 감정 분류
from deepface import DeepFace
try:
    analysis = DeepFace.analyze(img_path=IMG_PATH, actions=["emotion"], enforce_detection=False)
    emo = analysis[0]["emotion"]
    dominant = analysis[0]["dominant_emotion"]
    print("😀 감정 분석:")
    for k,v in sorted(emo.items(), key=lambda x:-x[1]):
        print(f"  {k:10s}: {v:5.1f}%")
    print(f"\n지배적 감정: {dominant}")
except Exception as e:
    print("감정 분석 실패:", e)

**✔ 실현성 체크포인트**
- blendshape 값이 표정 변화에 실제로 반응하는가? (긴장한 표정 vs 웃는 표정 이미지 각각 테스트)
- DeepFace 감정 분류가 그럴듯한가?
- 실서비스: **랜드마크/blendshape는 브라우저 MediaPipe JS**로 실시간 렌더링, 감정 분류만 서버에서.

---
## 4️⃣ (보너스) rPPG 심박 추정 — "심박이 뛴다!" 연출

웹캠 영상의 얼굴 피부색 미세 변화로 심박수를 추정합니다. 정확도는 완벽하지 않지만
마피아 게임 **긴장 연출**로는 충분합니다. 아래는 원리 검증용 간이 구현입니다.

> 실서비스에서 제대로 하려면 `pyVHR` 같은 전용 라이브러리를 쓰세요. 여기선 개념 증명만.

In [ ]:
# 짧은 얼굴 영상(5~15초, 정면 고정) 업로드
from google.colab import files
up = files.upload()
VIDEO_PATH = list(up.keys())[0]
print("업로드됨:", VIDEO_PATH)

In [ ]:
# 간이 rPPG: 이마 ROI의 녹색 채널 평균 시계열 → 밴드패스 → 우세 주파수
import cv2, numpy as np
from scipy.signal import butter, filtfilt

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

signal = []
while True:
    ok, frame = cap.read()
    if not ok: break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.2, 5)
    if len(faces):
        x,y,w,h = faces[0]
        roi = frame[y:y+int(h*0.25), x+int(w*0.3):x+int(w*0.7)]  # 이마
        if roi.size: signal.append(roi[:,:,1].mean())  # G채널
cap.release()

signal = np.array(signal)
print(f"프레임 {len(signal)}개, fps≈{fps:.0f}")

if len(signal) > fps*3:
    sig = signal - signal.mean()
    lo, hi = 0.7/(fps/2), 3.5/(fps/2)   # 42~210 bpm
    b,a = butter(3, [lo,hi], btype="band")
    filt = filtfilt(b,a,sig)
    freqs = np.fft.rfftfreq(len(filt), 1/fps)
    ps = np.abs(np.fft.rfft(filt))**2
    bpm = freqs[np.argmax(ps)]*60
    print(f"💓 추정 심박수(연출용): {bpm:.0f} bpm")
    print("   (정면 고정·밝은 조명일수록 정확. 오락 연출용)")
else:
    print("⚠️ 영상이 너무 짧거나 얼굴 검출 실패. 5초 이상 정면 영상으로 재시도.")

---
## ✅ 최종 실현성 판단 체크리스트

노트북을 돌려본 뒤 아래를 팀과 함께 체크하세요:

| 기능 | 검증 질문 | 판단 |
|------|-----------|------|
| **STT** | 한국어 전사 품질이 로그로 쓸 만한가? 화자 분리를 트랙 분리로 해결 가능한가? | ☐ |
| **로그 LLM** | 조간신문/심판/진술 출력이 게임에 쓸 만한가? 비용은? | ☐ |
| **표정 분석** | blendshape·감정이 표정에 실제 반응하는가? | ☐ |
| **동요/심박 지수** | "연출용"으로 재미있는가? (과학적 진실판정 아님을 UI에 명시) | ☐ |

### 아키텍처 결론
- **AI = Python(FastAPI) 서비스**, 게임 로직 = Spring Boot, 실시간 얼굴 = 브라우저 MediaPipe JS.
- 가장 먼저 확정할 것: **`Utterance` 로그 스키마** (STT·LLM 세 기능이 전부 여기 의존).
- 가장 리스크 큰 것: 실시간 웹캠 파이프라인(WebRTC↔서버) 지연. → 초기에 PoC 권장.

### ⚠️ 윤리/법적 유의
표정·심박 기반 지표는 **오락용 연출**로만 사용하고, '거짓말 탐지'로 광고하지 마세요.
생체정보(얼굴·음성) 수집은 개인정보보호법 대상이므로 **동의 절차**를 반드시 설계하세요.
